# Export Anndata for Seurat Interoperability

In this notebook, we export anndata from the Xenium and scRNAseq references for import to R for Seurat label transfer.

Here, we load a a single processed scRNAseq reference adata object  from `/processed_adata`. THis contains neuron and non-neuron cell types from the following references, processed in `/notebooks/00b_download_references`:
- https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE139088
- https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE254789


**Pinned Environment:** [`envs/sc-scvi.yaml`](../.../envs/sc-scvi.yaml)  

In [1]:
import os
from pathlib import Path
import pandas as pd
import scanpy as sc
import scipy.sparse
from scipy.sparse import csr_matrix
import pandas as pd
import scipy.sparse
from scipy.io import mmwrite
from pathlib import Path
import zipfile, gzip, tempfile
import session_info

In [2]:
base_dir = Path('/home/workspace/projects/drg')

adata_dir = base_dir / "data/h5ad/export_02/02b_leiden"
ref_data_dir = '/home/workspace/private/projects/kim/drg/data/scrna-seq/h5ad/05_reference' # Concatenated dataset

# New unified export root
export_root = base_dir / "data/rds"

# Two separate dataset-specific directories
adata_outdir = export_root / "xenium"
refdata_outdir = export_root / "refdata"

output_dirs = [adata_outdir, refdata_outdir]
print(output_dirs)

[PosixPath('/home/workspace/projects/drg/data/rds/xenium'), PosixPath('/home/workspace/projects/drg/data/rds/refdata')]


In [3]:
adata = sc.read_h5ad(os.path.join(adata_dir, 'adata-leiden-475.h5ad'))
refdata = sc.read_h5ad(os.path.join(ref_data_dir, 'adata-reference.h5ad'))

adata_list = [adata, refdata]
dataset_names = ['adata', 'refdata']

## Prepare data
Make sure we're recovering all 475 genes and not losing them due to differences in nomenclature

In [4]:
genes_only_in_adata = set(adata.var_names) - set(refdata.var_names)
print(len(genes_only_in_adata))
genes_only_in_adata = sorted(list(genes_only_in_adata))
genes_only_in_adata[:20]   # preview first 20

4


['Ccn1', 'Pclaf', 'Tafa1', 'Tafa5']

In [5]:
# Rename aliases in 2020 dataset

aliases = {
    "Ccn1": "Cyr61",
    "Pclaf": "Kiaa0101",
    "Tafa1": "Fam19a1",
    "Tafa5": "Fam19a5"
}

for gene, alias in aliases.items():
    print(f"\nChecking {gene} and alias {alias}:")

    print("  adata:")
    print(f"    {gene}  > {gene in adata.var_names}")
    print(f"    {alias} > {alias in adata.var_names}")

    print("  refdata:")
    print(f"    {gene}  > {gene in refdata.var_names}")
    print(f"    {alias} > {alias in refdata.var_names}")


Checking Ccn1 and alias Cyr61:
  adata:
    Ccn1  > True
    Cyr61 > False
  refdata:
    Ccn1  > False
    Cyr61 > True

Checking Pclaf and alias Kiaa0101:
  adata:
    Pclaf  > True
    Kiaa0101 > False
  refdata:
    Pclaf  > False
    Kiaa0101 > False

Checking Tafa1 and alias Fam19a1:
  adata:
    Tafa1  > True
    Fam19a1 > False
  refdata:
    Tafa1  > False
    Fam19a1 > True

Checking Tafa5 and alias Fam19a5:
  adata:
    Tafa5  > True
    Fam19a5 > False
  refdata:
    Tafa5  > False
    Fam19a5 > True


In [6]:
old_names = ["Cyr61", "Fam19a1", "Fam19a5"]

for name in old_names:
    print("In refdata", name, ":", name in refdata.var_names)

In refdata Cyr61 : True
In refdata Fam19a1 : True
In refdata Fam19a5 : True


In [7]:
# mapping: what refdata HAS > what we WANT it to be
rename_map = {
    "Cyr61": "Ccn1",
    "Fam19a1": "Tafa1",
    "Fam19a5": "Tafa5"
}

# apply renaming safely
refdata.var.index = refdata.var.index.to_series().replace(rename_map)

# ensure AnnData internal references are updated
refdata.var_names_make_unique()

In [8]:
genes_only_in_adata = set(adata.var_names) - set(refdata.var_names)
print(len(genes_only_in_adata))
genes_only_in_adata = sorted(list(genes_only_in_adata))
genes_only_in_adata[:20]   # preview first 20

1


['Pclaf']

In [9]:
for ad in adata_list:
    if "counts" in ad.layers:
        ad.X = ad.layers["counts"].copy()
    else:
        raise ValueError(f"'counts' layer missing in object with shape {ad.shape}")

## Export components

In [10]:
layer_name = "counts"

for ad, outdir in zip(adata_list, output_dirs):

    output_zip = outdir / f"{layer_name}_matrix.zip"
    outdir.mkdir(parents=True, exist_ok=True)

    # Ensure var index has a name
    gene_id_key = ad.var.index.name or "gene_name"
    ad.var.index.name = gene_id_key

    # Features table (ensembl_id, gene_name, feature_type)
    gene_df = ad.var.reset_index()
    genes = gene_df[[gene_id_key]].copy()
    genes["gene_name"] = gene_df[gene_id_key].values
    genes["feature_type"] = "Gene Expression"

    # Barcodes
    barcodes = pd.DataFrame(ad.obs.index)

    # Cell metadata
    cell_metadata = ad.obs.reset_index()
    cell_metadata.rename(columns={"index": "barcode"}, inplace=True)

    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_path = Path(tmp_dir)

        # Matrix
        if layer_name in ad.layers:
            X = ad.layers[layer_name].T
        else:
            X = ad.X.T

        with gzip.open(tmp_path / "matrix.mtx.gz", "wb") as f:
            mmwrite(f, scipy.sparse.csc_matrix(X))

        # Features
        genes.to_csv(
            tmp_path / "features.tsv.gz",
            sep="\t", index=False, header=False, compression="gzip"
        )

        # Barcodes
        barcodes.to_csv(
            tmp_path / "barcodes.tsv.gz",
            sep="\t", index=False, header=False, compression="gzip"
        )

        # Cell metadata
        cell_metadata.to_csv(tmp_path / "cell_metadata.csv", index=False)

        # Zip output
        with zipfile.ZipFile(output_zip, "w") as zipf:
            for fname in ["matrix.mtx.gz", "features.tsv.gz", "barcodes.tsv.gz", "cell_metadata.csv"]:
                zipf.write(tmp_path / fname, arcname=fname)

    print(output_zip)

/home/workspace/projects/drg/data/rds/xenium/counts_matrix.zip
/home/workspace/projects/drg/data/rds/refdata/counts_matrix.zip


In [11]:
# EXPORT METADATA

for ad, outdir in zip(adata_list, output_dirs):

    save_dir = outdir / "metadata"
    save_dir.mkdir(parents=True, exist_ok=True)

    ad.obs.to_csv(save_dir / 'cell-metadata.csv')
    ad.var.to_csv(save_dir / 'feature-metadata.csv')

    print(f"Exported metadata to: {save_dir}")

Exported metadata to: /home/workspace/projects/drg/data/rds/xenium/metadata
Exported metadata to: /home/workspace/projects/drg/data/rds/refdata/metadata


In [12]:
# EXPORT DIMENSIONAL REDUCTIONS

latent_key_map = {
    "xenium": "X_scVI_ModelA_475",
    "refdata": "X_scVI"
}

for ad, outdir in zip(adata_list, output_dirs):

    dataset_name = outdir.name
    save_dir = outdir / "reductions"
    save_dir.mkdir(parents=True, exist_ok=True)

    # PCA
    if "X_pca" in ad.obsm:
        pca = pd.DataFrame(ad.obsm["X_pca"], index=ad.obs_names)
        pca.columns = ["PC_" + str(i+1) for i in range(pca.shape[1])]
        pca.to_csv(save_dir / "pca.csv")
        print("Exported PCA:", save_dir / "pca.csv")

    # LATENT
    if dataset_name in latent_key_map:
        latent_key = latent_key_map[dataset_name]

        if latent_key in ad.obsm:
            latent = pd.DataFrame(ad.obsm[latent_key], index=ad.obs_names)
            latent.columns = ["latent_" + str(i+1) for i in range(latent.shape[1])]
            latent.to_csv(save_dir / "latent_representation.csv")
            print("Exported latent (" + latent_key + "):", save_dir / "latent_representation.csv")
        else:
            print("Latent key", latent_key, "not found for", dataset_name)
    else:
        print("No latent key mapping for dataset:", dataset_name)

    # UMAP
    if "X_umap" in ad.obsm:
        umap = pd.DataFrame(ad.obsm["X_umap"], index=ad.obs_names)
        umap.columns = ["umap_" + str(i+1) for i in range(umap.shape[1])]
        umap.to_csv(save_dir / "umap.csv")
        print("Exported UMAP:", save_dir / "umap.csv")

Exported latent (X_scVI_ModelA_475): /home/workspace/projects/drg/data/rds/xenium/reductions/latent_representation.csv
Exported UMAP: /home/workspace/projects/drg/data/rds/xenium/reductions/umap.csv
Exported latent (X_scVI): /home/workspace/projects/drg/data/rds/refdata/reductions/latent_representation.csv
Exported UMAP: /home/workspace/projects/drg/data/rds/refdata/reductions/umap.csv
